# PATCHR Server on Google Colab

Run the PATCHR server on Colab's free GPU and get a public URL to connect from **PATCHR-Studio**.

Supports **Boltz2** (with inpainting) and **Protenix** models.

**Steps:**
1. Go to **Runtime > Change runtime type > GPU** (T4)
2. Run all cells below in order
3. Cell 4 downloads model weights (~4.5GB) — runs once, then cached
4. Cell 5 starts the server and opens a tunnel
5. Copy the public URL into PATCHR-Studio

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Install PATCHR

In [ ]:
!git clone https://github.com/DeepFoldProtein/patchr.git
%cd patchr
!pip install -e . -q

## 3. Install cloudflared

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
!cloudflared --version

## 4. Download model weights

Pick which model to load at startup:
- **`boltz2`** — Boltz2 with inpainting (default)
- **`protenix`** — Protenix
- **`all`** — Both (uses more VRAM)

Model weights (~4.5GB) are downloaded using fast parallel download (aria2c), then cached.
Optionally enable Google Drive cache for persistence across sessions.

In [ ]:
import subprocess
import time
import os
import urllib.request
import ssl
from pathlib import Path

# ── Choose model ──────────────────────────────────────────
MODEL = "boltz2"  # @param ["boltz2", "protenix", "all"]
USE_GDRIVE_CACHE = False  # @param {type:"boolean"}
# ──────────────────────────────────────────────────────────

# ── Fast pre-download model weights ───────────────────────
# aria2c (16 parallel connections) is preferred for speed.
# Falls back to urllib if aria2c is not available.

BOLTZ_CACHE = Path.home() / ".boltz"

# Optional: Google Drive cache (persistent across sessions)
if USE_GDRIVE_CACHE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    gdrive_cache = Path("/content/drive/MyDrive/.boltz_cache")
    gdrive_cache.mkdir(parents=True, exist_ok=True)
    if BOLTZ_CACHE.exists() and not BOLTZ_CACHE.is_symlink():
        import shutil
        for f in BOLTZ_CACHE.iterdir():
            dest = gdrive_cache / f.name
            if not dest.exists():
                shutil.move(str(f), str(dest))
        shutil.rmtree(BOLTZ_CACHE)
    if not BOLTZ_CACHE.exists():
        BOLTZ_CACHE.symlink_to(gdrive_cache)
    print(f"Using Google Drive cache: {gdrive_cache}")

BOLTZ_CACHE.mkdir(parents=True, exist_ok=True)

FILES = {
    "mols.tar": "https://huggingface.co/boltz-community/boltz-2/resolve/main/mols.tar",
    "boltz2_conf.ckpt": "https://huggingface.co/boltz-community/boltz-2/resolve/main/boltz2_conf.ckpt",
    "ccd.pkl": "https://huggingface.co/boltz-community/boltz-1/resolve/main/ccd.pkl",
}

# Check aria2c availability; try to install on Colab
HAS_ARIA2 = subprocess.run(["which", "aria2c"], capture_output=True).returncode == 0
if not HAS_ARIA2:
    r = subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], capture_output=True)
    HAS_ARIA2 = r.returncode == 0

if HAS_ARIA2:
    print("Using aria2c (parallel download)")
else:
    print("aria2c not available — using urllib (single-thread)")

# SSL context for urllib fallback
_ssl_ctx = ssl.create_default_context()
try:
    import certifi
    _ssl_ctx.load_verify_locations(certifi.where())
except ImportError:
    _ssl_ctx.check_hostname = False
    _ssl_ctx.verify_mode = ssl.CERT_NONE


def download_file(url, dest):
    """Download url to dest using aria2c if available, else urllib."""
    if HAS_ARIA2:
        result = subprocess.run(
            ["aria2c", "-x", "16", "-s", "16", "-k", "1M",
             "--file-allocation=none", "--console-log-level=warn",
             "-d", str(dest.parent), "-o", dest.name, url],
        )
        return result.returncode == 0 and dest.exists()
    else:
        try:
            urllib.request.urlretrieve(url, str(dest), context=_ssl_ctx)
            return dest.exists()
        except TypeError:
            # older Python: urlretrieve doesn't accept context
            opener = urllib.request.build_opener(
                urllib.request.HTTPSHandler(context=_ssl_ctx)
            )
            with opener.open(url) as resp, open(dest, "wb") as f:
                while True:
                    chunk = resp.read(1024 * 1024)
                    if not chunk:
                        break
                    f.write(chunk)
            return dest.exists()
        except Exception as e:
            print(f"    Download error: {e}")
            return False


need_extract_mols = False
for fname, url in FILES.items():
    fpath = BOLTZ_CACHE / fname
    if fname == "mols.tar":
        if (BOLTZ_CACHE / "mols").exists():
            print(f"  {fname}: already extracted, skipping")
            continue
    if fpath.exists():
        print(f"  {fname}: already cached")
        if fname == "mols.tar":
            need_extract_mols = True
        continue

    print(f"  {fname}: downloading...")
    t0 = time.time()
    ok = download_file(url, fpath)
    if ok:
        elapsed = time.time() - t0
        size_mb = fpath.stat().st_size / 1e6
        speed = size_mb / elapsed if elapsed > 0 else 0
        print(f"  {fname}: {size_mb:.0f}MB in {elapsed:.0f}s ({speed:.1f} MB/s)")
    else:
        print(f"  WARNING: Failed to download {fname}")

    if fname == "mols.tar" and fpath.exists():
        need_extract_mols = True

# Extract mols.tar if needed
if need_extract_mols and (BOLTZ_CACHE / "mols.tar").exists():
    mols_dir = BOLTZ_CACHE / "mols"
    if not mols_dir.exists():
        print("  Extracting mols.tar...")
        t0 = time.time()
        subprocess.run(["tar", "xf", str(BOLTZ_CACHE / "mols.tar"), "-C", str(BOLTZ_CACHE)])
        print(f"  Extracted in {time.time()-t0:.0f}s")

print("\nDownload complete!")

## 5. Start server & tunnel

Starts the PATCHR server with pre-cached model weights, waits for model loading, then opens a Cloudflare tunnel.

In [ ]:
import subprocess
import threading
import re
import time
import requests

# ── Start server ──────────────────────────────────────────
server_process = subprocess.Popen(
    ["patchr", "serve", "--model", MODEL, "--port", "31212", "--device-id", "0"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

def stream_server_logs():
    for line in server_process.stdout:
        text = line.decode("utf-8", errors="ignore").rstrip()
        if text:
            print(f"[server] {text}")
log_thread = threading.Thread(target=stream_server_logs, daemon=True)
log_thread.start()

print(f"Starting PATCHR server (model={MODEL})...")
print("Model data is pre-cached — startup should be fast.\n")
server_ready = False
model_ready = False

# Phase 1: Server HTTP ready
for i in range(60):
    if server_process.poll() is not None:
        print(f"\nERROR: Server process exited with code {server_process.returncode}")
        break
    time.sleep(2)
    try:
        r = requests.get("http://localhost:31212/api/v1/health", timeout=2)
        if r.status_code == 200:
            server_ready = True
            print(f"Server is accepting requests (took ~{(i+1)*2}s)")
            break
    except Exception:
        pass
else:
    print("\nWARNING: Server did not respond within 120s.")

if not server_ready:
    print("Cannot start tunnel — server is not running.")
else:
    # Phase 2: Wait for model GPU loading (should be fast since files are cached)
    print("Loading model to GPU...")
    last_stage = ""
    for i in range(180):
        if server_process.poll() is not None:
            print(f"\nERROR: Server process crashed during model loading.")
            break
        time.sleep(5)
        try:
            r = requests.get("http://localhost:31212/api/v1/health", timeout=3)
            if r.status_code == 200:
                health = r.json()
                if not health.get("model_loading", False):
                    model_ready = True
                    models = health.get("loaded_models", [])
                    elapsed = (i + 1) * 5
                    print(f"Model loaded! ({models}) — took ~{elapsed}s")
                    break
                else:
                    stage = health.get("loading_stage", "loading...")
                    elapsed = (i + 1) * 5
                    if stage != last_stage:
                        print(f"  [{elapsed}s] {stage}")
                        last_stage = stage
                    elif i % 12 == 0:
                        print(f"  [{elapsed}s] {stage}")
        except Exception:
            pass
    else:
        print("\nWARNING: Model did not finish loading within 900s.")

    if not model_ready:
        print("Model loading failed or timed out. Check the logs above.")

    # Start cloudflared tunnel
    tunnel_process = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", "http://localhost:31212", "--no-chunked-encoding"],
        stdout=subprocess.PIPE, stderr=subprocess.PIPE,
    )

    tunnel_url = None
    def read_output():
        global tunnel_url
        for line in tunnel_process.stderr:
            match = re.search(r"(https://[\w-]+\.trycloudflare\.com)", line.decode("utf-8", errors="ignore"))
            if match:
                tunnel_url = match.group(1)
                break
    t = threading.Thread(target=read_output, daemon=True)
    t.start()
    t.join(timeout=20)

    if tunnel_url:
        tunnel_ok = False
        for attempt in range(3):
            try:
                r = requests.get(f"{tunnel_url}/api/v1/health", timeout=10)
                if r.status_code == 200:
                    tunnel_ok = True
                    break
            except Exception:
                time.sleep(2)

        print("\n" + "=" * 60)
        print(f"  PATCHR Server is running!")
        print(f"  Model: {MODEL}")
        print(f"  Public URL: {tunnel_url}")
        if tunnel_ok:
            print(f"  Status: Verified (health check passed through tunnel)")
        else:
            print(f"  Status: WARNING — tunnel URL obtained but health check failed")
            print(f"          Try the health check cell below, it may need a moment")
        print(f"\n  Copy this URL into PATCHR-Studio")
        print("=" * 60)
    else:
        print("Failed to get tunnel URL. Try re-running this cell.")

## 6. Health check (optional)

Verify the server is reachable through the tunnel.

In [ ]:
import requests

try:
    r = requests.get(f"{tunnel_url}/api/v1/health", timeout=10)
    print("Health check:", r.json())
except Exception as e:
    print(f"Error: {e}")